# Local Key Data Demo

This notebook computes the key local data used in the presentation: dataset summary, EDA signals, model benchmark, ablation, tuning evidence, CV/public comparison, and XGBoost branch reconstruction.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / "code" / "ppt_key_data_check.py").exists() and (path / "data" / "train.csv").exists():
            return path
    raise FileNotFoundError("Open this notebook from inside the repository folder.")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "code"))

In [ ]:
import pandas as pd

train = pd.read_csv(PROJECT_ROOT / "data" / "train.csv")
test = pd.read_csv(PROJECT_ROOT / "data" / "test.csv")

group_id = train["PassengerId"].str[:4]
group_size = group_id.value_counts()
group_labels = train.assign(GroupId=group_id).groupby("GroupId")["Transported"]

dataset_summary = pd.DataFrame([
    ["Training rows", len(train)],
    ["Test rows", len(test)],
    ["Raw predictive columns", train.drop(columns=["Transported"]).shape[1]],
    ["Positive class rate", f"{train['Transported'].mean() * 100:.2f}%"],
    ["Largest missing rate", f"{train.isna().mean().max() * 100:.2f}%"],
    ["Multi-passenger row rate", f"{group_id.map(group_size).gt(1).mean() * 100:.1f}%"],
    ["Group-homogeneous rate", f"{group_labels.nunique().eq(1).mean() * 100:.2f}%"],
], columns=["Metric", "Value"])

dataset_summary

In [ ]:
cabin = train["Cabin"].str.split("/", expand=True)
eda = train.copy()
eda["Deck"] = cabin[0]
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
eda["TotalSpend"] = eda[spend_cols].fillna(0).sum(axis=1)
eda["NoSpend"] = eda["TotalSpend"].eq(0)

eda_signals = pd.DataFrame([
    ["Deck B transport rate", f"{eda.groupby('Deck')['Transported'].mean().loc['B'] * 100:.0f}%"],
    ["Deck T transport rate", f"{eda.groupby('Deck')['Transported'].mean().loc['T'] * 100:.0f}%"],
    ["Europa transport rate", f"{eda.groupby('HomePlanet')['Transported'].mean().loc['Europa'] * 100:.0f}%"],
    ["Earth transport rate", f"{eda.groupby('HomePlanet')['Transported'].mean().loc['Earth'] * 100:.0f}%"],
    ["No-spend rate if transported", f"{eda.groupby('Transported')['NoSpend'].mean().loc[True] * 100:.1f}%"],
    ["No-spend rate if not transported", f"{eda.groupby('Transported')['NoSpend'].mean().loc[False] * 100:.1f}%"],
], columns=["Signal", "Value"])

eda_signals

In [ ]:
tables = PROJECT_ROOT / "experiments" / "tables"

model_benchmark = pd.read_csv(tables / "model_benchmark.csv")
feature_ablation = pd.read_csv(tables / "feature_ablation.csv")
tuning_summary = pd.read_csv(tables / "tuning_summary.csv")
cv_vs_public = pd.read_csv(tables / "cv_vs_public.csv")
xgb_reconstruction = pd.read_csv(tables / "xgb_branch_reconstruction.csv")

model_benchmark

In [ ]:
feature_ablation

In [ ]:
tuning_summary

In [ ]:
cv_vs_public

In [ ]:
xgb_reconstruction

The Kaggle public-best branch is kept in `demo_081716_xgb.ipynb`. This local notebook focuses on the data and validation evidence used in the presentation.